# Day 1: E-Commerce Post-Dispatch RTO Risk Predictor — Dataset Exploration & ML Fundamentals

This notebook inspects the raw e-commerce order dataset (`data/raw/orders.csv`) and processed splits (`data/processed/`), audits data quality (nulls, duplicates, schemas), verifies the binary target distribution (`rto`), and establishes the feature selection strategy.


In [1]:
import pandas as pd
import numpy as np

# Load raw orders dataset
df = pd.read_csv('../data/raw/orders.csv')
print(f'Successfully loaded raw dataset with shape: {df.shape}')


Successfully loaded raw dataset with shape: (50000, 24)


In [2]:
print('Dataset Shape:', df.shape)
print('\nColumns List (24):', df.columns.tolist())
display(df.head(5))


Dataset Shape: (50000, 24)

Columns List (24): ['order_id', 'order_ts', 'customer_id', 'customer_name', 'phone', 'address_line', 'pincode', 'city', 'district', 'state', 'pincode_tier', 'category', 'order_value', 'order_value_band', 'discount_pct', 'payment_mode', 'is_cod', 'is_festive', 'is_alternate_address', 'delivery_days_est', 'signup_date', 'account_age_days', 'order_velocity_24h', 'rto']


,order_id,order_ts,customer_id,customer_name,phone,address_line,pincode,city,district,state,pincode_tier,category,order_value,order_value_band,discount_pct,payment_mode,is_cod,is_festive,is_alternate_address,delivery_days_est,signup_date,account_age_days,order_velocity_24h,rto
0,ORD000000,2024-01-01 10:23:40.346132186,CUST010422,Bhavika Soni,7164076517,"colony, Pune",410403,Pune,Pune,Maharashtra,metro,home_kitchen,1657.0,1000_plus,0.155,PREPAID,False,False,False,3.7,2023-09-21,102.43,0,0
1,ORD000001,2024-01-01 11:13:43.171313827,CUST001599,Kritika Mani,3348943013,"H No H.No. 762, bazar, Near Petrol Pump, Pune East, Pune",411019,Pune,Pune,Maharashtra,metro,accessories,1351.0,1000_plus,0.134,COD,True,False,True,2.2,2023-11-18,44.47,0,0
2,ORD000002,2024-01-01 12:47:09.518135117,CUST006797,Panini Bhargava,3429094010,"H No H.No. 581, Ben Circle, Near SBI Bank, Ranchi Puram, Ranchi",835103,Ranchi,Ranchi,Jharkhand,tier_1,accessories,189.0,under_500,0.259,COD,True,False,False,3.7,2023-12-31,1.53,0,1
3,ORD000003,2024-01-01 13:41:32.281843979,CUST013642,Barkha Bali,483414083,"H No 46, main road, Near SBI Bank, New Delhi Sector 5, New Delhi",110028,New Delhi,West Delhi,Delhi,metro,fashion,769.0,500_1000,0.143,PREPAID,False,False,True,4.0,2023-12-31,1.57,0,0
4,ORD000004,2024-01-01 14:28:26.047731700,CUST005550,Libni Nanda,9937145445,"gali no 4, Chennai Sector 5, Chennai",600019,Chennai,Chennai,Tamil Nadu,metro,fashion,1795.0,1000_plus,0.152,PREPAID,False,False,False,2.4,2023-11-30,32.60,0,0


In [3]:
print('--- Dataset Info ---')
df.info()

print('\n--- Null Values per Column ---')
null_counts = df.isnull().sum()
print(null_counts)
print(f'\nTotal missing values across entire dataset: {null_counts.sum()}')

print('\n--- Duplicate Rows ---')
dup_count = df.duplicated().sum()
print(f'Total duplicate rows: {dup_count}')


--- Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   order_id              50000 non-null  object 
 1   order_ts              50000 non-null  object 
 2   customer_id           50000 non-null  object 
 3   customer_name         50000 non-null  object 
 4   phone                 50000 non-null  int64  
 5   address_line          50000 non-null  object 
 6   pincode               50000 non-null  int64  
 7   city                  50000 non-null  object 
 8   district              50000 non-null  object 
 9   state                 50000 non-null  object 
 10  pincode_tier          50000 non-null  object 
 11  category              50000 non-null  object 
 12  order_value           50000 non-null  float64
 13  order_value_band      50000 non-null  object 
 14  discount_pct          50000 non-null  float64
 15

In [4]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Categorical / Object columns count: {len(cat_cols)}')
print('Categorical columns:', cat_cols)
print('=' * 80)

for col in cat_cols:
    n_unique = df[col].nunique()
    print(f'Column: {col} | Unique Values: {n_unique:,}')
    print('Top 5 values with counts:')
    print(df[col].value_counts().head(5))
    print('-' * 60)


Categorical / Object columns count: 13
Categorical columns: ['order_id', 'order_ts', 'customer_id', 'customer_name', 'address_line', 'city', 'district', 'state', 'pincode_tier', 'category', 'order_value_band', 'payment_mode', 'signup_date']
Column: order_id | Unique Values: 50,000
Top 5 values with counts:
order_id
ORD000000    1
ORD033350    1
ORD033328    1
ORD033329    1
ORD033330    1
Name: count, dtype: int64
------------------------------------------------------------
Column: order_ts | Unique Values: 44,590
Top 5 values with counts:
order_ts
2024-09-28 23:59:56.400000000    40
2024-10-06 23:59:56.400000000    39
2024-10-31 23:59:56.400000000    34
2024-11-05 23:59:56.400000000    34
2024-10-19 23:59:56.400000000    32
Name: count, dtype: int64
------------------------------------------------------------
Column: customer_id | Unique Values: 14,702
Top 5 values with counts:
customer_id
CUST008197    26
CUST001184    26
CUST015853    25
CUST003825    23
CUST017853    23
Name: count

In [5]:
# Target Variable Value Counts & Proportions
rto_counts = df['rto'].value_counts()
rto_pct = df['rto'].value_counts(normalize=True) * 100

target_summary = pd.DataFrame({
    'Count': rto_counts,
    'Percentage (%)': rto_pct.round(2),
    'Meaning': ['Delivered Successfully (Class 0)', 'Returned to Origin (RTO - Class 1)']
})
print(target_summary)


Count  Percentage (%)                             Meaning
rto                                                           
0    41815           83.63    Delivered Successfully (Class 0)
1     8185           16.37  Returned to Origin (RTO - Class 1)


In [6]:
# Payment Mode vs RTO
print('--- RTO by Payment Mode ---')
print(df.groupby('payment_mode')['rto'].agg(['count', 'mean']).rename(columns={'mean': 'rto_rate'}))

# Pincode Tier vs RTO
print('\n--- RTO by Pincode Tier ---')
print(df.groupby('pincode_tier')['rto'].agg(['count', 'mean']).rename(columns={'mean': 'rto_rate'}))

# Category vs RTO
print('\n--- RTO by Category ---')
print(df.groupby('category')['rto'].agg(['count', 'mean']).rename(columns={'mean': 'rto_rate'}))

# Festive Period vs RTO
print('\n--- RTO by Festive Season ---')
print(df.groupby('is_festive')['rto'].agg(['count', 'mean']).rename(columns={'mean': 'rto_rate'}))


--- RTO by Payment Mode ---
              count  rto_rate
payment_mode                 
COD           30908  0.256115
PREPAID       19092  0.014090

--- RTO by Pincode Tier ---
              count  rto_rate
pincode_tier                 
metro         14776  0.126083
tier_1        19870  0.145747
tier_2        10747  0.205174
tier_3         4607  0.265031

--- RTO by Category ---
              count  rto_rate
category                     
accessories    7108  0.138154
beauty         6564  0.131018
electronics    7355  0.062542
fashion       14883  0.277363
footwear       5502  0.156125
home_kitchen   8588  0.104332

--- RTO by Festive Season ---
            count  rto_rate
is_festive                 
False       36472  0.149402
True        13528  0.202247


In [7]:
train = pd.read_parquet('../data/processed/train.parquet')
val = pd.read_parquet('../data/processed/val.parquet')
test = pd.read_parquet('../data/processed/test.parquet')

print(f'Train split: {train.shape} | Order dates: {train["order_ts"].min().strftime("%Y-%m-%d")} to {train["order_ts"].max().strftime("%Y-%m-%d")} | RTO: {train["rto"].mean():.2%}')
print(f'Val split:   {val.shape} | Order dates: {val["order_ts"].min().strftime("%Y-%m-%d")} to {val["order_ts"].max().strftime("%Y-%m-%d")} | RTO: {val["rto"].mean():.2%}')
print(f'Test split:  {test.shape} | Order dates: {test["order_ts"].min().strftime("%Y-%m-%d")} to {test["order_ts"].max().strftime("%Y-%m-%d")} | RTO: {test["rto"].mean():.2%}')


Train split: (35000, 42) | Order dates: 2024-01-01 to 2025-01-22 | RTO: 16.96%
Val split:   (5000, 42) | Order dates: 2025-01-22 to 2025-03-16 | RTO: 15.50%
Test split:  (10000, 42) | Order dates: 2025-03-16 to 2025-06-30 | RTO: 14.73%
